# Практика: исключения и работа с файлами

На этом занятии:

- рассмотрим функцию `open`, режимы работы с файлом `r`, `w`, `a` и основные методы файловых объектов;
- познакомимся с конструкцией `with` как альтернативой `try-finally`;
- научимся работать с путями и файлами с помощью `pathlib.Path`;
- сохраним и прочитаем данные в формате JSON с помощью модуля `json`.

In [ ]:
import json
import time
from collections.abc import Callable
from functools import wraps
from pathlib import Path
from typing import Any

## 1. Практическая задача: декоратор `backoff`

### Что такое backoff

При обращении к внешним системам (сетевым сервисам, базам данных, удаленным хранилищам файлов) часть ошибок носит временный характер: сервис перезапускается, сеть кратковременно недоступна, превышен лимит запросов, база данных занята другой транзакцией. Повторный вызов через некоторое время нередко завершается успешно.

**Backoff** (*exponential backoff*, экспоненциальная задержка) — стратегия повторных попыток, при которой время ожидания перед каждой следующей попыткой увеличивается. Обычно задержка умножается на постоянный коэффициент, например `1 с`, `2 с`, `4 с`, `8 с`, а верхняя граница ограничивает максимальное время ожидания между попытками.

### Зачем нужен backoff

- Немедленные повторные вызовы без паузы создают дополнительную нагрузку на сервис, который и так не справляется с запросами, и, как правило, завершаются той же ошибкой. Увеличивающаяся пауза дает сервису время на восстановление.
- Ограничение числа попыток и верхняя граница задержки гарантируют, что вызывающий код не будет ждать бесконечно: когда попытки исчерпаны, исключение все равно возбуждается.
- Повторять имеет смысл только вызовы, завершившиеся временной ошибкой. Ошибка в аргументах (`ValueError`, `TypeError`) при повторе не исчезнет, поэтому типы исключений, при которых вызов повторяется, задаются явно.

### Где используется

Стратегия применяется в клиентах HTTP API (в том числе в SDK облачных провайдеров), драйверах баз данных, клиентах брокеров сообщений, при загрузке файлов по сети и взаимодействии микросервисов. В промышленном коде обычно используются готовые библиотеки (`backoff`, `tenacity`). К задержке там часто добавляют случайную составляющую (*jitter*), чтобы повторные запросы от множества клиентов не приходились на один и тот же момент времени. В задаче реализуем упрощенный вариант без случайной составляющей.

### Условие

Реализуйте параметрический декоратор `backoff` (см. [конспект о параметрических декораторах](../lesson08/conspects/09-parametrized-decorators.md)) со следующими параметрами:

- `scale` — коэффициент, во сколько раз увеличивается время ожидания после каждой неудачной попытки;
- `delay_start` — время ожидания (в секундах) после первой неудачной попытки;
- `delay_max` — максимальное время ожидания (в секундах) между попытками;
- `tries_count` — максимальное число попыток вызова функции, включая первую;
- `exceptions` — кортеж типов исключений, при возникновении которых вызов повторяется.

Декорированная функция ведет себя следующим образом:

1. Функция вызывается с переданными аргументами. Если вызов завершился без исключения, возвращается его результат.
2. Если возникло исключение, тип которого входит в `exceptions` (или является подклассом одного из этих типов), и использованы не все попытки, выполняется ожидание, после чего функция вызывается повторно с теми же аргументами. Время ожидания после первой неудачной попытки равно `delay_start`, после каждой следующей оно умножается на `scale`, но не превышает `delay_max`.
3. Если исключение возникло при последней попытке, оно возбуждается повторно без изменений. После последней неудачной попытки ожидание не выполняется.
4. Исключения, типы которых не входят в `exceptions`, не перехватываются: они сразу передаются вызывающему коду, повторные попытки не выполняются.
5. Каждый вызов декорированной функции начинается заново: с первой попытки и с времени ожидания `delay_start`.

Дополнительные требования:

- ожидание выполняется вызовом `time.sleep(delay)` (модуль `time` импортирован в начале ноутбука): проверка ниже заменяет эту функцию, чтобы не ждать реальное время;
- декорированная функция сохраняет имя и документацию исходной функции (`functools.wraps`).

Перед созданием декоратора, то есть при вызове `backoff(...)`, необходимо проверить параметры. Если проверка не пройдена, возбуждается `ValueError` с поясняющим сообщением:

- `delay_start` и `delay_max` должны быть положительными, а `delay_start` должен быть меньше `delay_max`;
- `tries_count` должно быть натуральным числом (целое число, большее 0).

### Пример

Для `scale=2`, `delay_start=1`, `delay_max=5`, `tries_count=5` и функции, которая каждый раз возбуждает исключение из `exceptions`:

| попытка | результат | ожидание после попытки |
|---|---|---|
| 1 | исключение | `1` с |
| 2 | исключение | `2` с |
| 3 | исключение | `4` с |
| 4 | исключение | `5` с (значение `8` ограничено `delay_max`) |
| 5 | исключение | нет, исключение возбуждается повторно |

```python
attempts: list[int] = []


@backoff(
    scale=2,
    delay_start=0.5,
    delay_max=5,
    tries_count=4,
    exceptions=(ConnectionError, TimeoutError),
)
def fetch_data() -> str:
    attempts.append(1)

    if len(attempts) < 3:
        raise ConnectionError("service is unavailable")

    return "data"


print(fetch_data())   # data (после ожиданий 0.5 с и 1 с)
print(len(attempts))  # 3
```

Некорректные параметры:

```python
backoff(scale=2, delay_start=5, delay_max=1, tries_count=3, exceptions=(OSError,))
# ValueError: delay_start must be less than delay_max
```

In [ ]:
def backoff(
    scale: float,
    delay_start: float,
    delay_max: float,
    tries_count: int,
    exceptions: tuple[type[Exception], ...],
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    # ваш код
    return lambda function: function

### Проверка

In [ ]:
recorded_delays: list[float] = []
default_parameters: dict[str, Any] = {
    "scale": 2,
    "delay_start": 1,
    "delay_max": 5,
    "tries_count": 5,
    "exceptions": (ConnectionError,),
}


def fake_sleep(seconds: float) -> None:
    recorded_delays.append(seconds)


def make_flaky_function(
    failures_count: int,
    error_type: type[Exception],
) -> tuple[Callable[..., str], list[tuple[Any, ...]]]:
    calls: list[tuple[Any, ...]] = []

    def flaky_function(*args: Any, **kwargs: Any) -> str:
        """Fail several times, then return a result."""
        calls.append((args, kwargs))

        if len(calls) <= failures_count:
            raise error_type(f"attempt {len(calls)} failed")

        return "ok"

    return flaky_function, calls


def decorate(function: Callable[..., Any], **overrides: Any) -> Callable[..., Any]:
    return backoff(**(default_parameters | overrides))(function)


def catch_error(function: Callable[[], Any], error_type: type[Exception]) -> Exception:
    try:
        function()

    except error_type as error:
        return error

    else:
        raise AssertionError(f"{error_type.__name__} expected")


def is_value_error_raised(**overrides: Any) -> bool:
    try:
        backoff(**(default_parameters | overrides))

    except ValueError:
        return True

    else:
        return False


def check_parameters_validation() -> None:
    invalid_overrides = [
        {"delay_start": 0},
        {"delay_start": -1},
        {"delay_max": 0},
        {"delay_max": -5},
        {"delay_start": 5, "delay_max": 5},
        {"delay_start": 10, "delay_max": 1},
        {"tries_count": 0},
        {"tries_count": -2},
        {"tries_count": 2.5},
        {"tries_count": "3"},
    ]

    for overrides in invalid_overrides:
        assert is_value_error_raised(**overrides), (
            f"ValueError expected for {overrides}"
        )

    assert not is_value_error_raised(), "valid parameters must be accepted"
    assert not is_value_error_raised(tries_count=1)
    assert not is_value_error_raised(delay_start=0.1, delay_max=0.2)
    assert callable(backoff(**default_parameters))


def check_retries() -> None:
    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(10, ConnectionError)
    error = catch_error(decorate(flaky_function), ConnectionError)
    assert str(error) == "attempt 5 failed", "the last exception must be re-raised"
    assert len(calls) == 5
    assert recorded_delays == [1, 2, 4, 5], recorded_delays

    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(2, ConnectionError)
    assert decorate(flaky_function)() == "ok"
    assert len(calls) == 3
    assert recorded_delays == [1, 2], recorded_delays

    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(0, ConnectionError)
    assert decorate(flaky_function)() == "ok"
    assert len(calls) == 1
    assert recorded_delays == []

    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(10, ConnectionError)
    catch_error(decorate(flaky_function, tries_count=1), ConnectionError)
    assert len(calls) == 1
    assert recorded_delays == []

    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(10, ConnectionError)
    catch_error(
        decorate(flaky_function, scale=1.5, delay_start=2, delay_max=10, tries_count=6),
        ConnectionError,
    )
    assert [round(delay, 6) for delay in recorded_delays] == [2, 3, 4.5, 6.75, 10]


def check_exceptions_selection() -> None:
    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(10, ValueError)
    catch_error(decorate(flaky_function), ValueError)
    assert len(calls) == 1, "exceptions outside of the tuple must not be retried"
    assert recorded_delays == []

    flaky_function, calls = make_flaky_function(2, TimeoutError)
    decorated_function = decorate(flaky_function, exceptions=(OSError, KeyError))
    assert decorated_function() == "ok", "subclasses must be retried"
    assert len(calls) == 3

    flaky_function, calls = make_flaky_function(1, KeyError)
    decorated_function = decorate(flaky_function, exceptions=(OSError, KeyError))
    assert decorated_function() == "ok"
    assert len(calls) == 2


def check_calls_independence_and_metadata() -> None:
    recorded_delays.clear()
    flaky_function, calls = make_flaky_function(10, ConnectionError)
    decorated_function = decorate(flaky_function)
    catch_error(decorated_function, ConnectionError)
    catch_error(decorated_function, ConnectionError)
    assert recorded_delays == [1, 2, 4, 5, 1, 2, 4, 5], recorded_delays

    flaky_function, calls = make_flaky_function(2, ConnectionError)
    assert decorate(flaky_function)("first", key=1) == "ok"
    assert calls == [(("first",), {"key": 1})] * 3, (
        "arguments must be passed on each try"
    )

    assert decorate(flaky_function).__name__ == "flaky_function"
    assert decorate(flaky_function).__doc__ == flaky_function.__doc__


original_sleep = time.sleep
time.sleep = fake_sleep

try:
    check_parameters_validation()
    check_retries()
    check_exceptions_selection()
    check_calls_independence_and_metadata()

finally:
    time.sleep = original_sleep

print("All tests passed")

## 2. Работа с файлами: функция `open`

Файл — именованная область данных на диске. Чтобы работать с файлом из программы, его необходимо открыть: функция `open` обращается к операционной системе и возвращает **файловый объект** — объект, методы которого читают данные из файла и записывают их в файл.

Основные параметры функции `open`:

- `file` — путь к файлу: строка или объект `Path` (см. раздел 4). Относительный путь отсчитывается от текущего рабочего каталога программы;
- `mode` — режим открытия файла (строка);
- `encoding` — кодировка текстового файла. Если параметр не указан, кодировка определяется платформой и настройками системы, поэтому одна и та же программа может по-разному читать один и тот же файл на разных компьютерах. Для переносимости кода рекомендуется всегда указывать `encoding="utf-8"`.

### Параметр `mode`

Рассмотрим три режима. По умолчанию файл открывается в текстовом режиме: данные читаются и записываются как строки (`str`).

| режим | назначение | если файла нет | если файл есть |
|---|---|---|---|
| `"r"` (по умолчанию) | чтение | `FileNotFoundError` | открывается для чтения с начала |
| `"w"` | запись | файл создается | **содержимое удаляется**, запись начинается с начала |
| `"a"` | дозапись | файл создается | содержимое сохраняется, новые данные добавляются в конец |

Существуют и другие режимы: например, `"x"` (создание нового файла, если файл уже существует — `FileExistsError`), `"b"` (бинарный режим), `"+"` (чтение и запись одновременно). Полный перечень приведен в [документации функции `open`](https://docs.python.org/3/library/functions.html#open).

### Запись: методы `write` и `close`

Метод `write(text)` записывает строку в файл и возвращает число записанных символов. Перевод строки автоматически не добавляется, его необходимо указывать явно (`"\n"`).

Метод `close()` закрывает файл: буферизованные данные сохраняются на диск, а ресурс операционной системы освобождается. Если файл не закрыть, данные могут не попасть на диск, а число одновременно открытых файлов ограничено. После закрытия файла обращение к его методам чтения и записи приводит к `ValueError`.

In [ ]:
file = open("notes.txt", mode="w", encoding="utf-8")
written_count = file.write("first line\n")
file.write("second line\n")
file.close()
print(written_count)  # 11

### Чтение: методы `read`, `readline`, `readlines`

- `read()` возвращает все оставшееся содержимое файла одной строкой. Вызов `read(size)` возвращает не более `size` символов;
- `readline()` возвращает следующую строку вместе с символом `\n` в конце (в конце файла — пустую строку);
- `readlines()` возвращает список всех оставшихся строк.

Файловый объект хранит текущую позицию: чтение продолжается с места, на котором остановилось предыдущее. После того как файл прочитан целиком, повторный вызов `read()` возвращает пустую строку.

In [ ]:
file = open("notes.txt", mode="r", encoding="utf-8")
content = file.read()
repeated_content = file.read()
file.close()
print(repr(content))           # 'first line\nsecond line\n'
print(repr(repeated_content))  # ''

In [ ]:
file = open("notes.txt", mode="r", encoding="utf-8")
first_line = file.readline()
other_lines = file.readlines()
file.close()
print(repr(first_line))  # 'first line\n'
print(other_lines)       # ['second line\n']

### Построчное чтение

Файловый объект является итерируемым: цикл `for` получает из него строки по одной. В отличие от `read()` и `readlines()`, файл не загружается в память целиком, поэтому такой способ подходит для больших файлов. Символ `\n` в конце строки при необходимости удаляется методом `rstrip`.

In [ ]:
file = open("notes.txt", mode="r", encoding="utf-8")

for line in file:
    print(line.rstrip("\n"))

file.close()
# first line
# second line

### Режим `a` и метод `writelines`

Режим `"a"` добавляет данные в конец файла, сохраняя прежнее содержимое. Метод `writelines(lines)` записывает в файл все строки итерируемого объекта; символы перевода строки он не добавляет.

In [ ]:
file = open("notes.txt", mode="a", encoding="utf-8")
file.write("third line\n")
file.writelines(["fourth line\n", "fifth line\n"])
file.close()
file = open("notes.txt", mode="r", encoding="utf-8")
print(len(file.readlines()))  # 5
file.close()

### Режим `w` очищает файл

При открытии существующего файла в режиме `"w"` его прежнее содержимое удаляется сразу, еще до первой записи.

In [ ]:
file = open("notes.txt", mode="w", encoding="utf-8")
file.write("only line\n")
file.close()
file = open("notes.txt", mode="r", encoding="utf-8")
print(repr(file.read()))  # 'only line\n'
file.close()

### Исключения при работе с файлами

Ошибки операций с файлами возбуждаются как подклассы `OSError` (см. [конспект об иерархии исключений](conspects/02-exception-hierarchy.md)). Наиболее частые:

- `FileNotFoundError` — файл (или каталог на пути к нему) не существует;
- `FileExistsError` — файл уже существует (например, при режиме `"x"` или создании каталога);
- `PermissionError` — недостаточно прав на операцию;
- `IsADirectoryError` — вместо файла указан каталог.

In [ ]:
try:
    file = open("missing.txt", mode="r", encoding="utf-8")

except FileNotFoundError as error:
    print(f"cannot open file: {error}")
    # cannot open file: [Errno 2] No such file or directory: 'missing.txt'

## 3. Конструкция `with`

В примерах выше файл закрывался вызовом `close()` в конце. Если между `open` и `close` возникнет исключение, вызов `close()` не выполнится, и файл останется открытым. Надежное решение — `try-finally` (см. [конспект о `try-finally`](conspects/10-try-finally.md)): блок `finally` выполняется при любом исходе блока `try`. Вызов `open` размещается перед `try`: если открыть файл не удалось, закрывать нечего.

In [ ]:
file = open("notes.txt", mode="r", encoding="utf-8")

try:
    content = file.read()

finally:
    file.close()

print(repr(content))  # 'only line\n'

Конструкция `with` выполняет ту же работу короче:

```python
with expression as name:
    ...
```

Python вычисляет выражение, связывает результат с именем `name`, выполняет инструкции тела `with` (строки с отступом под заголовком), а при выходе из тела **в любом случае** — после нормального завершения, после `return`, `break`, `continue` или исключения — автоматически выполняет завершающее действие объекта. Для файлового объекта это вызов `close()`. Таким образом, `with` — удобная альтернатива `try-finally`.

`with` работает не с любыми объектами, а только с **контекстными менеджерами**. Файловые объекты, возвращаемые функцией `open`, являются контекстными менеджерами. Что это такое и как создавать собственные контекстные менеджеры, рассмотрим на следующих занятиях.

In [ ]:
with open("notes.txt", mode="r", encoding="utf-8") as file:
    content = file.read()

print(repr(content))  # 'only line\n'
print(file.closed)    # True

Имя `file` после блока `with` остается доступным, но файл уже закрыт (атрибут `closed` равен `True`). Файл закрывается и тогда, когда внутри блока возникло исключение:

In [ ]:
try:
    with open("notes.txt", mode="r", encoding="utf-8") as file:
        raise RuntimeError("something went wrong")

except RuntimeError as error:
    print(f"{error}, file closed: {file.closed}")
    # something went wrong, file closed: True

Объект, который не является контекстным менеджером, в `with` использовать нельзя:

In [ ]:
with 42 as number:
    print(number)
# TypeError: 'int' object does not support the context manager protocol

Далее файлы открываются только с помощью `with`.

## 4. Модуль `pathlib`

Пути к файлам до сих пор записывались строками. У такого подхода есть недостатки: разные операционные системы используют разные разделители (`\` в Windows, `/` в Linux и macOS), а склеивание частей пути конкатенацией строк приводит к ошибкам. Модуль `pathlib` предоставляет класс `Path` — объект пути, в котором операции над путем и файлом доступны как методы и операторы. Класс `Path` импортирован в начале ноутбука.

### Составление путей: оператор `/`

Оператор `/` присоединяет к пути новую часть. Хотя бы один из операндов должен быть объектом `Path`, второй может быть строкой. Разделитель подбирается автоматически для текущей операционной системы.

In [ ]:
data_directory = Path("practice_data")
report_path = data_directory / "reports" / "summary.txt"

print(report_path)
# practice_data\reports\summary.txt (Windows)
# practice_data/reports/summary.txt (Linux, macOS)

### Части пути

- `name` — имя файла с расширением;
- `stem` — имя файла без расширения;
- `suffix` — расширение (с точкой);
- `parent` — родительская директория (также объект `Path`).

In [ ]:
print(report_path.name)           # summary.txt
print(report_path.stem)           # summary
print(report_path.suffix)         # .txt
print(report_path.parent)         # practice_data\reports (Windows)
print(report_path.parent.parent)  # practice_data

### Создание директорий и файлов

- `path.mkdir()` создает директорию. Параметр `parents=True` создает и все отсутствующие родительские директории (иначе возбуждается `FileNotFoundError`), параметр `exist_ok=True` подавляет `FileExistsError`, если директория уже существует;
- `path.touch()` создает пустой файл (если файл существует, обновляется время его изменения);
- `path.exists()`, `path.is_file()` и `path.is_dir()` проверяют, существует ли путь и является ли он файлом или директорией.

In [ ]:
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.touch()
print(report_path.exists())         # True
print(report_path.is_file())        # True
print(report_path.parent.is_dir())  # True

### Чтение и запись файлов

У объекта `Path` есть методы, которые открывают файл, выполняют операцию и закрывают его самостоятельно:

- `read_text(encoding=...)` возвращает все содержимое файла строкой;
- `write_text(text, encoding=...)` записывает строку в файл, как режим `"w"`: прежнее содержимое заменяется. Метод возвращает число записанных символов.

Для бинарных данных существуют методы `read_bytes()` и `write_bytes()`.

In [ ]:
report_path.write_text("Total: 42\nErrors: 0\n", encoding="utf-8")
print(report_path.read_text(encoding="utf-8"))
# Total: 42
# Errors: 0

Если требуется другой режим (например, дозапись) или построчная работа, используется метод `path.open(mode=..., encoding=...)`. Он принимает те же параметры, что и функция `open`, и подходит для `with`. Объект `Path` можно передать и в саму функцию `open(path, ...)`.

In [ ]:
with report_path.open(mode="a", encoding="utf-8") as report_file:
    report_file.write("Warnings: 3\n")

print(report_path.read_text(encoding="utf-8").splitlines())
# ['Total: 42', 'Errors: 0', 'Warnings: 3']

Прежде чем читать файл, можно проверить его существование методом `exists()` (подход LBYL) или обработать `FileNotFoundError` (подход EAFP). Об их сравнении — в [конспекте о LBYL и EAFP](conspects/09-lbyl-vs-eafp.md).

In [ ]:
missing_path = data_directory / "missing.txt"

try:
    missing_path.read_text(encoding="utf-8")

except FileNotFoundError:
    print(f"file {missing_path.name} not found")  # file missing.txt not found

Мы рассмотрели не все возможности модуля `pathlib`. Например, в нем есть методы для обхода содержимого директорий (`iterdir`, `glob`), переименования и перемещения файлов (`rename`), получения сведений о файле (`stat`) и другие. Подробнее — в [документации модуля `pathlib`](https://docs.python.org/3/library/pathlib.html).

## 5. Модуль `json`

**JSON** (*JavaScript Object Notation*) — текстовый формат обмена данными. Он используется в веб-API, конфигурационных файлах и при хранении структурированных данных. Синтаксис JSON похож на литералы Python. Модуль `json` стандартной библиотеки преобразует объекты Python в текст JSON и обратно.

Соответствие типов:

| Python | JSON |
|---|---|
| `dict` | объект (`{...}`) |
| `list`, `tuple` | массив (`[...]`) |
| `str` | строка |
| `int`, `float` | число |
| `True`, `False` | `true`, `false` |
| `None` | `null` |

### Функции `dumps` и `loads`

- `json.dumps(obj)` преобразует объект Python в строку с JSON;
- `json.loads(text)` преобразует строку с JSON в объект Python.

Буква `s` в названиях означает *string*: функции работают со строками.

In [ ]:
user = {"name": "Alice", "age": 25, "is_active": True, "email": None}

user_json = json.dumps(user)
print(user_json)
# {"name": "Alice", "age": 25, "is_active": true, "email": null}
print(type(user_json))  # <class 'str'>

restored_user = json.loads(user_json)
print(restored_user == user)  # True

### Параметры форматирования

- `indent` — число пробелов для отступов (текст разбивается на строки, удобные для чтения человеком);
- `ensure_ascii=False` — сохраняет символы, не входящие в ASCII, как есть. По умолчанию они записываются escape-последовательностями `\uXXXX`. Особенно важно для кириллицы.

In [ ]:
city = {"city": "Zürich", "population": 420000}

print(json.dumps(city))
# {"city": "Z\u00fcrich", "population": 420000}
print(json.dumps(city, ensure_ascii=False, indent=2))
# {
#   "city": "Zürich",
#   "population": 420000
# }

### Ограничения формата

- JSON не различает `list` и `tuple`: кортеж записывается как массив и читается обратно как список;
- ключами объекта в JSON могут быть только строки: ключи других типов (например, `int`) преобразуются в строки;
- не все объекты Python сериализуются: например, множество (`set`) приводит к `TypeError`.

In [ ]:
original = {"point": (1, 2), 1: "one"}
restored = json.loads(json.dumps(original))
print(restored)  # {'point': [1, 2], '1': 'one'}

In [ ]:
try:
    json.dumps({"numbers": {1, 2, 3}})

except TypeError as error:
    print(error)  # Object of type set is not JSON serializable

### Функции `dump` и `load`: работа с файлами

Функции без буквы `s` работают с файловыми объектами:

- `json.dump(obj, file)` записывает JSON в открытый для записи файл;
- `json.load(file)` читает JSON из открытого для чтения файла и возвращает объект Python.

Параметры форматирования (`indent`, `ensure_ascii`) принимаются и функцией `dump`. Файл JSON, как и любой текстовый файл, следует открывать с `encoding="utf-8"`.

In [ ]:
settings = {"theme": "dark", "font_size": 14, "recent_files": ["a.txt", "b.txt"]}
settings_path = data_directory / "settings.json"
settings_path.parent.mkdir(parents=True, exist_ok=True)

with settings_path.open(mode="w", encoding="utf-8") as settings_file:
    json.dump(settings, settings_file, indent=2)

with settings_path.open(mode="r", encoding="utf-8") as settings_file:
    loaded_settings = json.load(settings_file)

print(loaded_settings == settings)  # True
print(settings_path.read_text(encoding="utf-8"))
# {
#   "theme": "dark",
#   "font_size": 14,
#   "recent_files": [
#     "a.txt",
#     "b.txt"
#   ]
# }

Тот же результат можно получить через `dumps`/`loads` и методы `write_text`/`read_text` объекта `Path`:

In [ ]:
settings_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
loaded_settings = json.loads(settings_path.read_text(encoding="utf-8"))
print(loaded_settings == settings)  # True

### Ошибки чтения JSON

Если текст не соответствует синтаксису JSON, возбуждается `json.JSONDecodeError`. Это подкласс `ValueError`. Частые причины: одинарные кавычки вместо двойных, лишняя запятая, пустой файл.

In [ ]:
try:
    json.loads("{'name': 'Alice'}")

except json.JSONDecodeError as error:
    print(f"invalid JSON: {error}")
    # invalid JSON: Expecting property name enclosed in double quotes:
    # line 1 column 2 (char 1)

## 6. Удаление файлов, созданных в примерах

Примеры создали файл `notes.txt` и директорию `practice_data`. Для удаления используются метод `unlink()` (удаляет файл; `missing_ok=True` подавляет `FileNotFoundError`), метод `rmdir()` (удаляет только пустую директорию) и функция `shutil.rmtree` (удаляет директорию вместе с содержимым).

In [ ]:
import shutil

Path("notes.txt").unlink(missing_ok=True)
shutil.rmtree(data_directory, ignore_errors=True)
print(Path("notes.txt").exists(), data_directory.exists())  # False False